# 🛡️ NariRaksha SLM - Colab Training Notebook
**Multilingual Women's Safety Reasoning Dataset & Small Language Model**

> ⚠️ **Before running anything**: Set runtime to T4 GPU via `Runtime > Change runtime type > T4 GPU`

## Cell 1 — Clone Repository

In [ ]:
# Clone the NariRaksha repository
!git clone https://github.com/Vikhram-Labs/NariRaksha.git
%cd NariRaksha
!git log --oneline -5

## Cell 2 — Clean Environment Setup (Run Once, Then Restart Runtime)

In [ ]:
# Run the definitive setup script — handles CUDA path, wipes conflicts, installs clean stack
!python scripts/colab_setup.py

## ⚠️ RESTART RUNTIME NOW
`Runtime > Restart session` — Required after installing PyTorch.

## Cell 3 — Verify GPU & CUDA Setup

In [ ]:
%cd /content/NariRaksha

import os
# Fix CUDA library path in this session
_cuda_paths = ['/usr/local/cuda/lib64', '/usr/local/cuda-12.1/lib64', '/usr/lib/x86_64-linux-gnu']
_existing = ':'.join(p for p in _cuda_paths if os.path.exists(p))
os.environ['LD_LIBRARY_PATH'] = _existing + ':' + os.environ.get('LD_LIBRARY_PATH', '')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import bitsandbytes as bnb
print(f'bitsandbytes: {bnb.__version__}')

## Cell 4 — Generate Dataset (1K examples, no API needed)

In [ ]:
# Generate 1000 safe mock examples that pass all safety checks
# Use --target 5000 or 10000 for larger datasets
!python nariraksha/synthetic_generation.py --target 1000 --use_mock

# Verify the output
import json
with open('datasets/synthetic/nariraksha_synthetic.jsonl') as f:
    lines = f.readlines()
print(f'Total examples: {len(lines)}')
print('Sample:')
print(json.dumps(json.loads(lines[0]), indent=2))

## Cell 5 — Create Benchmark Splits

In [ ]:
!python benchmark/benchmark.py \
    --dataset datasets/synthetic/nariraksha_synthetic.jsonl \
    --out_dir benchmark/splits

## Cell 6 — Run QLoRA Training

In [ ]:
# Train with Qwen 2.5 3B using QLoRA (fits within T4 16GB VRAM)
# Switch to configs/gemma.yaml or configs/smollm.yaml for other models
!python training/train.py --config configs/qwen.yaml

## Cell 7 — Evaluate

In [ ]:
!python evaluation/evaluate.py \
    --model_path models/nariraksha-qwen3-3b \
    --test_data benchmark/splits/adversarial_benchmark.jsonl

## Cell 8 — Publish to HuggingFace Hub (Optional)

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_your_token_here'  # Replace with your token

!python scripts/publish_dataset.py \
    --dataset_dir datasets/synthetic \
    --repo_id your_org/NariRaksha-100K \
    --token $HF_TOKEN

!python scripts/publish_model.py \
    --model_dir models/nariraksha-qwen3-3b \
    --repo_id your_org/NariRaksha-Qwen-3B \
    --token $HF_TOKEN